In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(VGAM)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(piano)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(foreach)
    library(ggpubr)
    library(DelayedArray)
    library(rmcorr)
    library(ggh4x)
})

In [ ]:
estimate_cell_cycle = function(cds, g1s_markers, g2m_markers){
    cds_g1s = cds[rowData(cds)$gene_short_name %in% g1s_markers,] 
    aggregate_g1s_expression = counts(cds_g1s)
    aggregate_g1s_expression = t(t(aggregate_g1s_expression) / colData(cds_g1s)$Size_Factor)
    aggregate_g1s_expression = Matrix::colSums(aggregate_g1s_expression)

    cds_g2m = cds[rowData(cds)$gene_short_name %in% g2m_markers,] 
    aggregate_g2m_expression = counts(cds_g2m)
    aggregate_g2m_expression = t(t(aggregate_g2m_expression) / colData(cds_g2m)$Size_Factor)
    aggregate_g2m_expression = Matrix::colSums(aggregate_g2m_expression)
    cds$g1s_score = log(aggregate_g1s_expression+1)
    cds$g2m_score = log(aggregate_g2m_expression+1)
    cds$proliferation_index = log(aggregate_g1s_expression + aggregate_g2m_expression + 1)
    return(cds)
}

aggregate_gene_score = function(cds, gene_list=c("CCNB1", "CCNB2","CDK1"), name = "gene_list_score"){
    cds_gene_list_norm = cds[rowData(cds)$gene_short_name %in% gene_list,] 
    aggregate_marker_expression = counts(cds_gene_list_norm)
    aggregate_marker_expression = t(t(aggregate_marker_expression) / colData(cds_gene_list_norm)$Size_Factor)
    aggregate_marker_expression = Matrix::colSums(aggregate_marker_expression)
    aggregate_marker_expression = log(aggregate_marker_expression + 1)
    colData(cds)[[name]] = aggregate_marker_expression
    return(cds)
}

# Load data

In [ ]:
cds_VCS = readRDS("../data/cds_VCS_with_cell_type_labels_E15-15.5_1e-4_subclustered_scores_by_embryo_30_all_genes_pseudotime_final.RDS")

In [ ]:
metadata = as.data.frame(colData(cds_VCS))

umap_coordinates = reducedDims(cds_VCS)[['UMAP']]

metadata = metadata %>%
             mutate(UMAP1 = umap_coordinates[,1],
                    UMAP2 = umap_coordinates[,2], 
                    UMAP3 = umap_coordinates[,3])

In [ ]:
cds_VCS$UMAP1 = metadata$UMAP1
cds_VCS$UMAP2 = metadata$UMAP2
cds_VCS$UMAP3 = metadata$UMAP3

In [ ]:
all_correlations_df = metadata %>%
    filter(!is.na(Pseudotime_avg)) %>%
    filter(
        Phenotype %in% c("Area", "Ratio"),
        Marker %in% c("Nucleolin", "SC35")
    ) %>%
    group_by(Embryo, trajectory_label, Cluster_sub_2, Marker, Phenotype) %>%
    summarise(
        med_score = median(avg_score, na.rm = TRUE),
        med_pseudotime = median(Pseudotime_avg, na.rm = TRUE),
        .groups = "drop"
    ) %>%
    
    mutate(Embryo = as.factor(Embryo)) %>%
    group_by(trajectory_label, Marker, Phenotype) %>%
    add_tally() %>%

    filter(n_distinct(Embryo) > 1) %>%
    
    mutate(
        ranked_pseudotime = rank(med_pseudotime, ties.method = "average"),
        ranked_score = rank(med_score, ties.method = "average")
    ) %>%

    summarise(
        pearson_model = list(
            tryCatch(
                rmcorr(participant = Embryo, measure1 = med_pseudotime, measure2 = med_score, dataset = pick(everything())),
                error = function(e) list(r = NA_real_, p = NA_real_) # Return NAs if it fails
            )
        ),

        spearman_model = list(
            tryCatch(
                rmcorr(participant = Embryo, measure1 = ranked_pseudotime, measure2 = ranked_score, dataset = pick(everything())),
                error = function(e) list(r = NA_real_, p = NA_real_)
            )
        ),

        pearson_rmc     = pearson_model[[1]]$r,
        pearson_pvalue  = pearson_model[[1]]$p,

        spearman_rmc    = spearman_model[[1]]$r,
        spearman_pvalue = spearman_model[[1]]$p,

        .groups = "drop"
    ) %>%
    select(-pearson_model, -spearman_model) %>%

    mutate(
        suffix = ifelse(Phenotype == "Area", "abs", "rel"),
        marker = paste0(Marker, "-", suffix)
    ) %>%
    select(-suffix)

In [ ]:
all_correlations_df = all_correlations_df %>%
                            mutate(trajectory_label = recode(trajectory_label, 
                                    "Definitive_erythroid" = "Definitive erythroids",
                                    "GABAergic" = "GABAergic neurons",
                                    "Glutamatergic" = "Glutamatergic neurons",
                                    "Fibroblasts_Fap+" = "Fibroblasts (Fap+)"))

In [ ]:
color = colorRampPalette(c("navy", "white", "red"))(101)
breaks = seq(-1, 1, length.out = 101)

options(repr.plot.width=4.5, repr.plot.height=3.7, repr.plot.res=300)

cor_matrix <- all_correlations_df %>%
    select(trajectory_label, spearman_rmc, marker) %>%
    tidyr::spread(marker, spearman_rmc) %>%
    mutate(trajectory_label = gsub('_', ' ', trajectory_label)) %>%
    column_to_rownames("trajectory_label") %>%
    as.matrix()

pval_matrix <- all_correlations_df %>%
    select(trajectory_label, spearman_pvalue, marker) %>%
    tidyr::spread(marker, spearman_pvalue) %>%
    mutate(trajectory_label = gsub('_', ' ', trajectory_label)) %>%
    column_to_rownames("trajectory_label") %>%
    as.matrix()

sig_matrix <- ifelse(pval_matrix < 0.001, "***", 
              ifelse(pval_matrix < 0.01, "**", 
              ifelse(pval_matrix < 0.05, "*", "")))

sig_matrix[is.na(sig_matrix)] <- ""

## Figure 4A
ComplexHeatmap::pheatmap(
    cor_matrix, 
    cluster_cols = F, 
    cluster_rows = T, 
    
    color = color, 
    breaks = breaks, 
    
    display_numbers = sig_matrix,    
    fontsize_number = 12,          
    number_color = "white",      
    
    na_col = "grey80",
    
    border_color = "white",        
    cellwidth = 22,                
    cellheight = 20,               
    
    fontsize_row = 9, 
    fontsize_col = 9,
    #angle_col = 45,
    
    treeheight_row = 20,
    
    show_colnames = FALSE,
    heatmap_legend_param = list(legend_height = unit(4.5, "cm")) #direction = "horizontal")
)

In [ ]:
metadata = metadata %>%
    mutate(traj_label = gsub("_", " ", trajectory_label))

In [ ]:
differentiation_df = metadata %>%
       filter(Phenotype %in% c("Area", "Ratio"),
        Marker %in% c("Nucleolin", "SC35")) %>%
    group_by(trajectory_label, traj_label, 
             Experiment, Embryo, Cluster_sub_2, Marker, Phenotype) %>%
    summarise(med_score = median(avg_score), med_pseudotime = median(Pseudotime_avg)) %>%
    as.data.frame()

In [ ]:
plot_data <- differentiation_df %>%
  left_join(., all_correlations_df, by = c("Marker", "Phenotype", "trajectory_label")) %>%
  mutate(facet_title = paste0(trajectory_label, "\n(ρ = ", round(spearman_rmc, 2), ")"))

tail(plot_data, 2)

In [ ]:
plot_data <- plot_data %>%
    mutate(Phenotype = case_match(Phenotype, "Ratio" ~ "Fraction", .default = Phenotype),
           Phenotypes = paste0(Marker, ' ', Phenotype)) %>%
    mutate(trajectory_label = recode(trajectory_label, 
                "Definitive_erythroid" = "Definitive erythroids",
                "GABAergic" = "GABAergic neurons",
                "Glutamatergic" = "Glutamatergic neurons",
                "Fibroblasts_Fap+" = "Fibroblasts (Fap+)"))

In [ ]:
## Supplementary Figure 14

options(repr.plot.width=10, repr.plot.height=17, repr.plot.res=300)
ggplot(plot_data %>%
        mutate(trajectory_label = gsub('_', ' ', trajectory_label)),
    aes(x = med_pseudotime, y = med_score)) + 
    geom_point(size = 0.2) + 
    geom_label(aes(label = paste0("ρ = ", sprintf("%.2f", spearman_rmc))), 
              x = Inf, y = Inf, 
              hjust = 1.1, vjust = 1.5,    
              size = 3.5, 
              fill = "white",       
              label.size = NA,      
              label.padding = unit(0.2, "lines"), 
              parse = F) +
    facet_grid2(trajectory_label ~ Phenotypes, scales = "free", independent = "all") + 
    labs(x = "Pseudotime", y = "Median score") + 
    monocle3:::monocle_theme_opts() + 
    theme(
        strip.text.x = element_text(size = 13), 
        strip.text.y = element_text(size = 8.5, angle = 270),
        strip.background = element_blank(),
        
        axis.text = element_text(color="black", size = 10),
        axis.ticks = element_line(color = "black"),
        axis.title = element_text(color = "black", size = 13)
    )

In [ ]:
cc.genes = readRDS("../data/cc.genes.RDS")
cc.genes$s.genes = stringr::str_to_title(cc.genes$s.genes)
cc.genes$g2m.genes = stringr::str_to_title(cc.genes$g2m.genes)

In [ ]:
cds_VCS = estimate_cell_cycle(cds_VCS, g1s_markers = cc.genes$s.genes, g2m_markers = cc.genes$g2m.genes)

## Erythroids

In [ ]:
cds_erythroids = cds_VCS[, cds_VCS$trajectory_refined == "Definitive_erythroid"]
cds_erythroids = aggregate_gene_score(cds_erythroids, gene_list = c('Cd36'), name = 'Cd36')
cds_erythroids = aggregate_gene_score(cds_erythroids, gene_list = c('Slc4a1'), name = 'Slc4a1')
cds_erythroids = aggregate_gene_score(cds_erythroids, gene_list = c('Gypa'), name = 'Gypa')
cds_erythroids = aggregate_gene_score(cds_erythroids, gene_list = c('Hbb-bs', 'Hbb-bt', 'Hbb-bh1', 'Hbb-y'), 
                                      name = 'Hemoglobin')
metadata_erythroids = as.data.frame(colData(cds_erythroids))

In [ ]:
umap_coordinates = reducedDims(cds_erythroids)[['UMAP']]

metadata_erythroids = metadata_erythroids %>%
             mutate(UMAP1 = umap_coordinates[,1],
                    UMAP2 = umap_coordinates[,2], 
                    UMAP3 = umap_coordinates[,3])

In [ ]:
scores = metadata_erythroids %>%
    filter(Marker == "Nucleolin" & Phenotype == "Area") %>%
    distinct(trajectory_refined, Cluster_sub, .keep_all=T) %>%
    pull(avg_score)

display(quantile(scores, c(0.1, 0.15, 0.5, 0.85, 0.9)))

scores = metadata_erythroids %>%
    filter(Marker == "Nucleolin" & Phenotype == "Ratio") %>%
    distinct(trajectory_refined, Cluster_sub, .keep_all=T) %>%
    pull(avg_score)

quantile(scores, c(0.1, 0.15, 0.5, 0.85, 0.9))

In [ ]:
## Figure 4B

options(repr.plot.width=5, repr.plot.height=4, repr.plot.res=300)
plot_cells(cds_erythroids[, cds_erythroids$Marker == "Nucleolin" & cds_erythroids$Phenotype == "Ratio"],
           x = 2, y = 3,
           reduction_method = "UMAP", 
           color_cells_by = "avg_score", 
           group_label_size = 3,
           label_cell_groups = T,
           cell_size = 0.3, alpha = 0.5)  +
        scale_colour_gradient2(low = muted("blue"), mid = "grey90", high = muted("red"), 
                               midpoint = 0.14, limits = c(0.11, 0.17), oob=scales::squish) + 
        labs(color = "Nucleolar fraction") + 
        theme(aspect.ratio = 1)


In [ ]:
min_max_scale <- function(x) {
  (x - min(x, na.rm = TRUE)) / (max(x, na.rm = TRUE) - min(x, na.rm = TRUE))
}

get_loess_normalized <- function(df, x_col, y_col) {
  clean_df <- df[!is.na(df[[x_col]]) & !is.na(df[[y_col]]), ]
  
  fit <- loess(as.formula(paste(y_col, "~", x_col)), data = clean_df, span = 0.9)

  res <- data.frame(x = seq(min(clean_df[[x_col]]), max(clean_df[[x_col]]), length.out = 100))
  colnames(res)[1] <- x_col
  pred <- predict(fit, newdata = res, se = TRUE)
  
  y_fit <- pred$fit
  y_upper <- pred$fit + (1.96 * pred$se.fit)
  y_lower <- pred$fit - (1.96 * pred$se.fit)
  
  mn <- min(y_fit, na.rm = TRUE)
  mx <- max(y_fit, na.rm = TRUE)
  
  res$y_n   <- (y_fit - mn) / (mx - mn)
  res$y_u_n <- (y_upper - mn) / (mx - mn)
  res$y_l_n <- (y_lower - mn) / (mx - mn)
  
  return(res)
}

In [ ]:
base_data <- metadata_erythroids %>%
    group_by(Experiment, trajectory_refined, Embryo, Cluster_sub) %>%
    summarise(
        med_prolif = median(proliferation_index, na.rm = TRUE),
        pt_all = median(Pseudotime_avg, na.rm = TRUE),
        med_score = median(avg_score[Marker == "Nucleolin" & Phenotype == "Ratio"], na.rm = TRUE),
        pt_filtered = median(Pseudotime_avg[Marker == "Nucleolin" & Phenotype == "Ratio"], na.rm = TRUE),
        .groups = "drop"
    ) %>%
    filter(!is.na(med_score) & !is.na(med_prolif)) %>%
    mutate(
        norm_prolif = min_max_scale(med_prolif),
        norm_score = min_max_scale(med_score)
    )

pts_score <- base_data %>%
    group_by(pt_filtered) %>%
    summarise(col_score = median(norm_score, na.rm = TRUE), .groups = "drop") %>%
    mutate(col_score_norm = min_max_scale(col_score))

pts_prolif <- base_data %>%
    group_by(pt_all) %>%
    summarise(col_prolif = median(norm_prolif, na.rm = TRUE), .groups = "drop") %>%
    mutate(col_prolif_norm = min_max_scale(col_prolif))

l1_score <- get_loess_normalized(base_data, "pt_filtered", "norm_score")
l1_prolif <- get_loess_normalized(base_data, "pt_all", "norm_prolif")

## Figure 4C
options(repr.plot.width=3.5, repr.plot.height=3, repr.plot.res=300)
ggplot() + 
    geom_ribbon(data = l1_score, aes(x = pt_filtered, ymin = y_l_n, ymax = y_u_n), fill = "grey70", alpha = 0.4) + 
    geom_ribbon(data = l1_prolif, aes(x = pt_all, ymin = y_l_n, ymax = y_u_n), fill = "#fb9a99", alpha = 0.25) + 
    geom_line(data = l1_score, aes(x = pt_filtered, y = y_n), linewidth = 1.2, color = "black") + 
    geom_line(data = l1_prolif, aes(x = pt_all, y = y_n), linewidth = 1.2, color = "#fb9a99") + 
    geom_point(data = pts_score, aes(x = pt_filtered, y = col_score_norm), size = 1.5, shape = 3, color = "black") + 
    geom_point(data = pts_prolif, aes(x = pt_all, y = col_prolif_norm), size = 1.5, shape = 17, color = "black") + 
    labs(x = "Pseudotime", y = "Normalized score") + 
    monocle3:::monocle_theme_opts() + 
    theme(legend.position = "none", 
          axis.text = element_text(color="black", size = 11), 
          axis.ticks = element_line(color = "black"), 
          aspect.ratio = 1)

In [ ]:
# Find the exact x-axis value where the normalized y is at its maximum
peak_x_score <- loess_score_data$med_Pseudotime_filtered[which.max(loess_score_data$y_norm)]
peak_x_prolif <- loess_prolif_data$med_Pseudotime_all[which.max(loess_prolif_data$y_norm)]

# Print the results to the console
print(paste("Peak Nucleolin Pseudotime:", round(peak_x_score, 3)))
print(paste("Peak Proliferation Pseudotime:", round(peak_x_prolif, 3)))

## Secretory

In [ ]:
secretory_lineages = c("Osteoblasts", "Chondrocytes", "Fibroblasts_Fap+", "Hepatocytes")

In [ ]:
## Figure 4G

options(repr.plot.width=3, repr.plot.height=3, repr.plot.res=300)
metadata %>%
    filter(trajectory_label %in% secretory_lineages) %>%
    filter(Marker == "Nucleolin" & Phenotype == "Ratio") %>%
    group_by(nn_majority_cell_type_umap) %>%
    mutate(Pseudotime_avg = min_max_scale(Pseudotime_avg)) %>%
    group_by(trajectory_label, nn_majority_cell_type_umap, Embryo, Cluster_sub_2, Marker) %>%
    summarise(med_score = median(avg_score), med_pseudotime = median(Pseudotime_avg)) %>%
    ggplot(., aes(x = med_pseudotime, y = med_score, color = nn_majority_cell_type_umap)) + 
        #geom_point() + 
        labs(x = "Pseudotime", y = "Nucleolar fraction score") + 
        geom_smooth(method = "lm") +
        scale_color_manual(values = c("#66c2a5", "#fc8d62", "#8da0cb", "black")) + 
        #facet_wrap(~trajectory_refined, scales="free") + 
        monocle3:::monocle_theme_opts() + theme(legend.position = "none") + 
        theme(axis.text.x = element_text(color="black", size = 10), 
              axis.text.y = element_text(color = "black", size = 10),
              axis.ticks = element_line(color = "black")) + 
    theme(aspect.ratio = 1)

# Fig 4 UMAP marker expression

In [ ]:
cds_VCS_nuc_frac = cds_VCS[, cds_VCS$Marker == "Nucleolin" &
                             cds_VCS$Phenotype == "Ratio"]

cds_VCS_nuc_area = cds_VCS[, cds_VCS$Marker == "Nucleolin" &
                             cds_VCS$Phenotype == "Area"]

cds_VCS_sc35_area = cds_VCS[, cds_VCS$Marker == "SC35" &
                             cds_VCS$Phenotype == "Area"]

In [ ]:
cds_osteo = cds_VCS_nuc_frac[, cds_VCS_nuc_frac$trajectory_label == "Osteoblasts"]
cds_chondro = cds_VCS_nuc_frac[, cds_VCS_nuc_frac$trajectory_label == "Chondrocytes"]
cds_hepato = cds_VCS_nuc_frac[, cds_VCS_nuc_frac$trajectory_label == "Hepatocytes" &
                                !is.na(cds_VCS_nuc_frac$Pseudotime_avg)]
cds_fap = readRDS("../data/cds_fibroblasts_fap.RDS")


In [ ]:
display(quantile(unique(cds_chondro$avg_score), c(0.1, 0.15, 0.5, 0.85, 0.9)))
display(quantile(unique(cds_fap$avg_score), c(0.1, 0.15, 0.5, 0.85, 0.9)))
display(quantile(unique(cds_hepato$avg_score), c(0.1, 0.15, 0.5, 0.85, 0.9)))
display(quantile(unique(cds_osteo$avg_score), c(0.1, 0.15, 0.5, 0.85, 0.9)))

In [ ]:
## Figure 4E

options(repr.plot.width=4.5, repr.plot.height=3, repr.plot.res=300)
plot_cells(cds_chondro, scale_to_range = F,
           x = 2, y = 3, min_expr = 0.1,
           reduction_method = "UMAP", 
           color_cells_by = "avg_score", 
           group_label_size = 3,
           label_cell_groups = F,
           cell_size = 0.4, alpha = 0.7) +
        scale_colour_gradient2(low = muted("blue"), mid = "grey90", high = muted("red"), 
                               midpoint = 0.104, limits = c(0.09, 0.118), oob=scales::squish) + 
        labs(color = "Nucleolar fraction") + 
        theme(aspect.ratio = 1)

options(repr.plot.width=4.5, repr.plot.height=3, repr.plot.res=300)
plot_cells(cds_chondro, scale_to_range = F,
           genes = c("Mef2c"),
           x = 2, y = 3, min_expr = 0.3,
           reduction_method = "UMAP", 
           color_cells_by = "avg_score", 
           group_label_size = 3,
           label_cell_groups = F,
           cell_size = 0.5, alpha = 1) +
        labs(color = "Nucleolar fraction") + 
        theme(aspect.ratio = 1)

In [ ]:
## Figure 4F

options(repr.plot.width=4.5, repr.plot.height=3, repr.plot.res=300)
plot_cells(cds_fap[, cds_fap$Marker == "Nucleolin" &
                     cds_fap$Phenotype == "Ratio"], 
           scale_to_range = F, show_trajectory_graph = F,
           x = 1, y = 2, min_expr = 0.1,
           reduction_method = "UMAP", 
           color_cells_by = "avg_score", 
           group_label_size = 3,
           label_cell_groups = F,
           cell_size = 0.4, alpha = 0.7) +
        scale_colour_gradient2(low = muted("blue"), mid = "grey90", high = muted("red"), 
                               midpoint = 0.103, limits = c(0.095, 0.111), oob=scales::squish) + 
        labs(color = "Nucleolar fraction") + 
        theme(aspect.ratio = 1)

options(repr.plot.width=4.5, repr.plot.height=3, repr.plot.res=300)
plot_cells(cds_fap, scale_to_range = F, show_trajectory_graph = F,
           genes = c("Fap"),
           x = 1, y = 2, min_expr = 1.3,
           reduction_method = "UMAP", 
           color_cells_by = "avg_score", 
           group_label_size = 3,
           label_cell_groups = F,
           cell_size = 0.5, alpha = 1) +
        labs(color = "Nucleolar fraction") + 
        theme(aspect.ratio = 1)

In [ ]:
## Figure 4G

options(repr.plot.width=4.5, repr.plot.height=3, repr.plot.res=300)
plot_cells(cds_hepato, scale_to_range = F,
           x = 1, y = 3, min_expr = 0.1,
           reduction_method = "UMAP", 
           color_cells_by = "avg_score", 
           group_label_size = 3,
           label_cell_groups = F,
           cell_size = 0.4, alpha = 0.7) +
        scale_colour_gradient2(low = muted("blue"), mid = "grey90", high = muted("red"), 
                               midpoint = 0.154, limits = c(0.14, 0.168), oob=scales::squish) + 
        labs(color = "Nucleolar fraction") + 
        theme(aspect.ratio = 1)

options(repr.plot.width=4.5, repr.plot.height=3, repr.plot.res=300)
plot_cells(cds_hepato, scale_to_range = F,
           genes = c("Cps1"),
           x = 1, y = 3, min_expr = 0.2,
           reduction_method = "UMAP", 
           color_cells_by = "avg_score", 
           group_label_size = 3,
           label_cell_groups = F,
           cell_size = 0.4, alpha = 1) +
        labs(color = "Nucleolar fraction") + 
        theme(aspect.ratio = 1)

In [ ]:
## Figure 4H

options(repr.plot.width=4.5, repr.plot.height=3, repr.plot.res=300)
plot_cells(cds_osteo, scale_to_range = F,
           x = 2, y = 3, min_expr = 0.1,
           reduction_method = "UMAP", 
           color_cells_by = "avg_score", 
           group_label_size = 3,
           label_cell_groups = F,
           cell_size = 0.4, alpha = 0.7) +
        scale_colour_gradient2(low = muted("blue"), mid = "grey90", high = muted("red"), 
                               midpoint = 0.118, limits = c(0.11, 0.126), oob=scales::squish) + 
        labs(color = "Nucleolar fraction") + 
        theme(aspect.ratio = 1)

options(repr.plot.width=4.5, repr.plot.height=3, repr.plot.res=300)
plot_cells(cds_osteo, scale_to_range = F,
           genes = c("Alpl"),
           x = 2, y = 3, min_expr = 0.2,
           reduction_method = "UMAP", 
           color_cells_by = "avg_score", 
           group_label_size = 3,
           label_cell_groups = F,
           cell_size = 0.4, alpha = 1) +
        labs(color = "Nucleolar fraction") + 
        theme(aspect.ratio = 1)